# Filtering in the Frequency Domain — Problem Statement


## Context

Frequency-domain processing represents an image as a superposition of spatial frequencies. Correct filtering requires understanding the 2-D DFT, spectrum interpretation, transfer functions, reconstruction, ringing, periodic noise, and quantitative validation.


## Problem Statement

Develop a reproducible Fourier-domain workflow that explains image spectra and applies low-pass, high-pass, band, notch, and illumination-related filters while connecting every frequency-domain operation to its spatial effect.


## Inputs and Fixed Parameters

Use module-local datasets under `../data/`, NumPy FFT conventions with explicit centering, repository-relative paths, real-valued reconstructed images after inverse transforms, and `../outputs/figures/` for diagnostics.


## 1. Spatial Frequency

A sinusoidal brightness pattern can be written as:

$$
g(x)=A\sin(2\pi f x+\phi)
$$

where:

- $A$ = amplitude;
- $f$ = spatial frequency;
- $\phi$ = phase.

Low spatial frequency means intensity changes slowly across space.  
High spatial frequency means intensity changes rapidly.

**Important:** high frequency does not mean high brightness.


## 2. Sinusoids, Complex Numbers, and the DFT

The DFT of a 1-D signal is:

$$
X[k]
=
\sum_{n=0}^{N-1}
x[n]e^{-j2\pi kn/N}
$$

Inverse:

$$
x[n]
=
\frac{1}{N}
\sum_{k=0}^{N-1}
X[k]e^{j2\pi kn/N}
$$

Euler's identity:

$$
e^{j\theta}
=
\cos(\theta)+j\sin(\theta)
$$

For $X=a+jb$:

$$
|X|=\sqrt{a^2+b^2}
$$

and

$$
\phi=\operatorname{atan2}(b,a)
$$

Magnitude = frequency strength.  
Phase = spatial alignment.


## 3. The 2-D Fourier Transform for Images

For image $f(x,y)$:

$$
F(u,v)
=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The FFT computes the DFT efficiently.

`fftshift` moves the zero-frequency component to the center:

- center → low frequencies;
- farther from center → high frequencies.

Before inverse FFT, undo the shift with `ifftshift`.


## 4. Reading a 2-D Spectrum

A useful orientation rule:

> Spatial stripes produce spectral energy perpendicular to the stripe direction.

Let's prove it visually.


## 5. Inverse FFT and Reconstruction

The Fourier transform is reversible if we keep all coefficients.


## 6. Magnitude vs Phase

Every Fourier coefficient can be written as:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

Magnitude tells us how strong a frequency is.  
Phase strongly controls spatial organization.


## 7. Frequency-Domain Filtering

Let $F$ be the image spectrum and $H$ the filter:

$$
G(u,v)=H(u,v)F(u,v)
$$

Then:

$$
g(x,y)=\mathcal{F}^{-1}\{G(u,v)\}
$$

Workflow:

1. FFT;
2. center with `fftshift`;
3. construct $H$;
4. multiply $H\cdot F$;
5. undo shift;
6. IFFT;
7. keep the real component.


## 8. Frequency Distance Grid

For circular filters:

$$
D(u,v)=
\sqrt{(u-u_0)^2+(v-v_0)^2}
$$


## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters

### Ideal LPF

$$
H(u,v)=
\begin{cases}
1,&D(u,v)\le D_0\\
0,&D(u,v)>D_0
\end{cases}
$$

### Gaussian LPF

$$
H(u,v)
=
\exp\left(
-\frac{D(u,v)^2}{2D_0^2}
\right)
$$

### Butterworth LPF

$$
H(u,v)
=
\frac{1}
{1+\left(\frac{D(u,v)}{D_0}\right)^{2n}}
$$

Butterworth order $n$ controls transition steepness.


## 10. Ringing and the Gibbs Phenomenon

A hard spectral cutoff corresponds to an oscillatory spatial response:

> abrupt spectral boundary → spatial oscillations → halos near edges


## 11. High-Pass Filtering

For a normalized LPF:

$$
H_{HP}=1-H_{LP}
$$

High frequencies contain edges and fine detail, but can also contain noise.


## 12. High-Boost Sharpening

A pure high-pass result mainly contains detail.

For sharpening:

$$
g(x,y)=f(x,y)+k f_{HP}(x,y)
$$


## 13. Convolution Theorem

$$
f*h
\quad\Longleftrightarrow\quad
F\cdot H
$$

Spatial convolution corresponds to multiplication in the frequency domain.

### Circular vs Linear Convolution

A DFT assumes periodic extension.

Therefore direct FFT multiplication naturally performs **circular convolution**.  
For ordinary linear convolution, appropriate zero-padding is generally required.


## 14. Band-Pass and Band-Reject Filters

Band-pass keeps:

$$
D_1\le D(u,v)\le D_2
$$

Band-reject removes that interval.


## 15. Periodic Noise

Periodic interference is one of the strongest reasons to use the frequency domain.

Repeated interference often becomes isolated off-center peaks in the spectrum.


## 16. Spectral Peak Detection

The following detector is intentionally simple:

1. remove the central low-frequency area;
2. rank remaining coefficients;
3. keep strong points separated by a minimum distance.


## 17. Notch-Reject Filtering

A notch-reject filter suppresses a small neighborhood around selected unwanted frequencies.

Real images have conjugate-symmetric spectra, so corresponding symmetric frequencies must also be considered.


## 18. Moiré Removal

Moiré is a repeated interference pattern. It can often be easier to isolate in the Fourier domain than in the spatial domain.


## 19. Slowly Varying Illumination / Shading

A simple multiplicative model is:

$$
I(x,y)\approx R(x,y)L(x,y)
$$

where:

- $R$ = reflectance / useful structure;
- $L$ = slowly varying illumination.

Because illumination varies slowly, it is dominated by low frequencies.


## 20. Cutoff Sensitivity

For a low-pass filter:

- smaller cutoff → stronger smoothing;
- larger cutoff → more detail preserved.


## 21. Quantitative Checks

MSE:

$$
\mathrm{MSE}
=
\frac{1}{MN}
\sum_{x,y}
[f(x,y)-g(x,y)]^2
$$

PSNR:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

PSNR measures numerical fidelity to a reference. It is not a universal perceptual-quality metric.


## 22. Validation Checks

Implement and validate this frequency-domain processing stage.


## 23. Common Mistakes

1. Displaying raw FFT magnitude instead of `log1p(abs(F))`.
2. Forgetting `fftshift`.
3. Forgetting `ifftshift`.
4. Confusing brightness with frequency.
5. Assuming all high frequencies are noise.
6. Using Ideal filters without expecting ringing.
7. Removing bright peaks blindly.
8. Ignoring circular convolution and zero-padding.
9. Treating PSNR as universal perceptual quality.
10. Memorizing formulas without linking spectrum to spatial structure.


## 24. Practical Exercises

### Beginner

1. Generate 4, 8, 16, and 32-cycle gratings.
2. Predict FFT peak locations before running.
3. Compare vertical, horizontal, and diagonal stripes.
4. Verify FFT → IFFT reconstruction.

### Intermediate

5. Test $D_0\in\{10,20,40,80\}$.
6. Test Butterworth $n\in\{1,2,4,8\}$.
7. Compare ringing on a binary square.
8. Repeat magnitude/phase swapping with another image pair.
9. Explain when HPF also amplifies noise.

### Advanced

10. Choose notch locations manually for `astronaut-interference.tif`.
11. Compare manual notches with automatic peak candidates.
12. Tune notch radius.
13. Remove moiré from `car-moire-pattern.tif`.
14. Tune shading correction on `text-spotshade.tif`.
15. Add zero-padding and demonstrate circular vs linear convolution.


## 25. Interview / Exam Questions

**What does the Fourier transform represent?**  
2-D spatial-frequency content as complex coefficients.

**Why use `fftshift`?**  
To center zero frequency for interpretation and filter design.

**Why use log magnitude?**  
FFT magnitude has a very large dynamic range.

**Magnitude vs phase?**  
Magnitude gives component strength; phase carries spatial alignment and much structural information.

**Core filtering equation?**

$$
G(u,v)=H(u,v)F(u,v)
$$

**Why does Ideal LPF ring?**  
Its abrupt cutoff corresponds to an oscillatory spatial response.

**What does Butterworth order control?**  
Transition steepness.

**Why is periodic noise often easier in Fourier domain?**  
It can appear as localized spectral peaks.

**Why symmetric peak pairs?**  
Real-valued images have conjugate-symmetric Fourier spectra.

**Convolution theorem?**  
Spatial convolution corresponds to frequency multiplication.

**Why can FFT filtering wrap around?**  
The DFT assumes periodic extension and naturally gives circular convolution.


## 26. Final Concept Map

```text
SPATIAL IMAGE
     │
     ├── smooth variation ─────────► low frequencies
     ├── edges / detail ───────────► high frequencies
     └── periodic patterns ────────► spectral peaks
                         │
                         ▼
                       FFT2
                         │
                      fftshift
                         │
             ┌───────────┴───────────┐
             │                       │
         MAGNITUDE                 PHASE
       frequency strength      spatial organization
             │                       │
             └───────────┬───────────┘
                         │
                    H(u,v) × F
                         │
                     ifftshift
                         │
                       IFFT2
                         │
                         ▼
                  FILTERED IMAGE
```


## Completion Criterion

The Implementation notebook must execute end-to-end, reconstruct images correctly, generate the required spectral/spatial diagnostics, and pass its numerical validation checks.
